# Vacancy formation energy in bcc Fe (EAM)

Builds a Fe supercell from `bulk('Fe', a=2.85, cubic=True)`, removes
one atom, and computes `E_vac - E_bulk` (the bare difference) under
the included Al-Fe.eam.fs EAM potential.

Driven from a `Workflow` so that `engine.with_working_directory(...)`
is evaluated eagerly with the resolved engine value.


In [1]:
import pathlib
from ase.build import bulk
from ase.calculators.eam import EAM

from pyiron_workflow import Workflow
from pyiron_workflow_atomistics.engine import ASEEngine, CalcInputMinimize, calculate
from pyiron_workflow_atomistics.structure import (
    create_supercell_with_min_dimensions,
    create_vacancy,
)
from pyiron_workflow_atomistics.physics.point_defect import (
    calculate_vacancy_formation_energy,
)

engine = ASEEngine(
    EngineInput=CalcInputMinimize(
        force_convergence_tolerance=0.05, max_iterations=100
    ),
    calculator=EAM(potential=str(next(p for p in [pathlib.Path("Al-Fe.eam.fs"), pathlib.Path("notebooks/Al-Fe.eam.fs"), pathlib.Path(__file__).parent / "Al-Fe.eam.fs" if "__file__" in globals() else pathlib.Path(".")] if p.exists()))),
    working_directory="./_vac_runs",
)


In [2]:
# Pre-build the supercell so we know N at workflow-construction time —
# the corrected formula E_f = E_vac - ((N-1)/N) * E_bulk needs N as a scalar.
fe_supercell = create_supercell_with_min_dimensions.node_function(
    bulk("Fe", a=2.85, cubic=True), min_dimensions=[7, 7, 7]
)
n_atoms_supercell = len(fe_supercell)
print(f"supercell: {n_atoms_supercell} Fe atoms")

wf = Workflow("vacancy_formation_energy")
wf.supercell = create_supercell_with_min_dimensions(
    bulk("Fe", a=2.85, cubic=True), min_dimensions=[7, 7, 7]
)
wf.with_vacancy = create_vacancy(wf.supercell, remove_atom_index=0)
wf.calc_bulk = calculate(
    wf.supercell, engine=engine.with_working_directory("supercell"), label="calc_bulk"
)
wf.calc_vac = calculate(
    wf.with_vacancy, engine=engine.with_working_directory("vacancy"), label="calc_vac"
)
wf.E_form = calculate_vacancy_formation_energy(
    vacancy_energy=wf.calc_vac.outputs.engine_output.final_energy,
    supercell_energy=wf.calc_bulk.outputs.engine_output.final_energy,
    n_atoms_supercell=n_atoms_supercell,
)
wf.run()

e_f = wf.E_form.outputs.formation_energy.value
print(f"Vacancy formation energy of Fe in BCC Fe (Al-Fe.eam.fs): {e_f:.3f} eV")

supercell: 54 Fe atoms


      Step     Time          Energy          fmax
BFGS:    0 17:56:08     -216.690138        0.000000


      Step     Time          Energy          fmax
BFGS:    0 17:56:08     -210.847774        0.368379


BFGS:    1 17:56:09     -210.864488        0.330939


BFGS:    2 17:56:10     -210.942616        0.131664


BFGS:    3 17:56:11     -210.947698        0.119259


BFGS:    4 17:56:12     -210.961581        0.038241


Vacancy formation energy of Fe in BCC Fe (Al-Fe.eam.fs): 1.716 eV
